In [1]:
from instance_generator import instance
K = 5
T_d = 10
C_k = [10] * K
done = False
instance = instance(T_d, K)

### State space: 
$\mathcal{S} = (t, \hat{C}, r, q)$ with $t \in \N$, $\hat{C} \in \{0,1,\dots,C_1\} \times \dots \times \{0,1,\dots,C_K\} \in \N^K$, $r \in \R$ and $q \in \N^K$,   
such that $t$ refers to the actual period, $C$ is the vector of remaining capacitities in each time slot and the pair $[r,q]$ is the next request to schedule.
  
Intital state: $S_1 = (1,C_1, \dots, C_K, r_1, q_1)$

### Actions:
Receiving a new request $[r_i,q_i]$:  
We can either decline the request which leads to a revenue of $0$ or we can accept the request, which yields to a revenue of $r_i$.  
Accepting the request we need to do an additional decision where to place the order. There are $K-len(q_i)+1$ different possibilities on how to schedule the request.  
There are $K+1$ different actions ($\mathcal{A} = \{0,1,\dots,K\}$) with the following transition function $\mathcal{T}: \mathcal{S} \times {\mathcal{A}} \rightarrow \mathcal{S}$. Let $S_i = (t,\hat{C}_1, \dots \hat{C}_K)$  
  
$0$: Decline the order --> $S_{i+1} = (t+1, \hat{C}_1, \dots \hat{C}_K, r_{i+1}, q_{i+1})$  
$1$: Accept the order on time slot 1  --> $S_{i+1} = (t+1, \hat{C}_1 - q_i^1, \dots, \hat{C}_{len(q_i)} - q_i^{len(q_i)}, \dots \hat{C}_K, r_{i+1}, q_{i+1})$  
$\vdots$  
$K$: Accept the order on time slot K --> $S_{i+1} = (t+1, \hat{C}_1 , \dots, \hat{C}_K - q_i^1, r_{i+1}, q_{i+1})$  
  
We need to pay attention to not allow accepting the order on time slot $l$ s.t. $l + len(q_i) > K$. We do this by violating this case with a highly negative reward.

We need to define a transition function $\mathcal{S} \times \mathcal{A} \rightarrow \mathcal{S}$ and a reward function $\mathcal{S} \times \mathcal{A} \rightarrow \R$

In [ ]:
def length_of_requests(state, K):
    q = state[K+2:] 
    for i, val in enumerate(q):
        if val == 0:
            return i
    return len(q)

def transition(state, action):
    len_q = length_of_requests(state, K)
    if len_q + action - 1 > K: # End of trajectory
        print("End of trajectory")
        done = True
        reward = -100
        return state, reward, done
    if action == 0:
        new_state = [state[0]+1] + state[1:K+1] + [instance[state[0]][0]] + instance[state[0]][1:K+1]
        reward = 0
    else:
        new_state = [state[0] +1] + state[1:action] + [state[action + i] - instance[state[0]-1][i+1] for i in range(len_q)] + state[action + len_q : K + 1] + [instance[state[0]][0]] + instance[state[0]][1:K+1]
        reward = instance[state[0]-1][0]
    if all(c >= 0 for c in new_state[1:K+1]):
        return new_state, reward, False
    else:
        done = True
        return new_state, -100, done


In [42]:
instance

[[14.190683258912223, 1, 1, 1, 0, 0],
 [12.413828940750877, 2, 4, 4, 0, 0],
 [14.995493493130983, 4, 2, 3, 2, 0],
 [11.665601037279435, 3, 0, 0, 0, 0],
 [14.446659906479695, 2, 2, 3, 1, 4],
 [12.160733139079563, 2, 3, 0, 0, 0],
 [13.33595079876067, 2, 3, 0, 0, 0],
 [7.287970645452906, 1, 1, 0, 0, 0],
 [6.625833755968287, 3, 1, 3, 3, 0],
 [3.287403214406382, 2, 2, 1, 2, 3]]

In [ ]:
state_0 = [1] + C_k + [instance[0][0]] + instance[0][1:K+1]
print(state_0)
(state_1, reward, done) = transition(state_0, 4)
print(state_1)
print(reward)
print(done)

[1, 10, 10, 10, 10, 10, 14.190683258912223, 1, 1, 1, 0, 0]
End of trajectory
[1, 10, 10, 10, 10, 10, 14.190683258912223, 1, 1, 1, 0, 0]
-100
True
